In [6]:
# pip install opencv-python face_recognition numpy pandas

# !pip install cmake
# !pip install dlib
# !pip install face_recognition

In [ ]:
import cv2
import face_recognition
import numpy as np
import pandas as pd
from datetime import datetime
import os


path = 'images'
images = []
classNames = []

if not os.path.exists(path):
    print("'images' folder not found")
    exit()

myList = os.listdir(path)

if len(myList) == 0:
    print("No images found in 'images' folder")
    exit()

for cl in myList:
    curImg = cv2.imread(f'{path}/{cl}')

    if curImg is not None:
        images.append(curImg)
        classNames.append(os.path.splitext(cl)[0])

print("Loaded Names:", classNames)


def findEncodings(images):
    encodeList = []

    for img in images:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        enc = face_recognition.face_encodings(img)

        if len(enc) > 0:
            encodeList.append(enc[0])

    return encodeList


encodeListKnown = findEncodings(images)

if len(encodeListKnown) == 0:
    print("No faces found in images")
    exit()

print("Encoding Complete")


def markAttendance(name):
    file = 'attendance.csv'

    if not os.path.exists(file):
        df = pd.DataFrame(columns=["Name", "Date", "Time"])
        df.to_csv(file, index=False)

    df = pd.read_csv(file)

    today = datetime.now().strftime('%Y-%m-%d')

    if ((df['Name'] == name) & (df['Date'] == today)).any():
        return

    now = datetime.now()
    timeString = now.strftime('%H:%M:%S')

    new_entry = pd.DataFrame(
        [[name, today, timeString]],
        columns=["Name", "Date", "Time"]
    )

    df = pd.concat([df, new_entry], ignore_index=True)
    df.to_csv(file, index=False)


cap = cv2.VideoCapture(1)

if not cap.isOpened():
    print("Camera can't access")

    img = np.zeros((300, 500, 3), dtype=np.uint8)

    cv2.putText(
        img,
        "Camera Can't Access",
        (50, 150),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 0, 255),
        2
    )

    cv2.imshow("Error", img)
    cv2.waitKey(3000)

    exit()


while True:
    success, img = cap.read()

    if not success:
        print("Failed to read from camera")
        break

    imgS = cv2.resize(img, (0, 0), None, 0.25, 0.25)
    imgS = cv2.cvtColor(imgS, cv2.COLOR_BGR2RGB)

    facesCurFrame = face_recognition.face_locations(imgS)
    encodesCurFrame = face_recognition.face_encodings(
        imgS,
        facesCurFrame
    )

    for encodeFace, faceLoc in zip(
        encodesCurFrame,
        facesCurFrame
    ):

        matches = face_recognition.compare_faces(
            encodeListKnown,
            encodeFace
        )

        faceDis = face_recognition.face_distance(
            encodeListKnown,
            encodeFace
        )

        matchIndex = np.argmin(faceDis)

        if matches[matchIndex]:
            name = classNames[matchIndex].upper()
        else:
            name = "UNKNOWN"

        y1, x2, y2, x1 = faceLoc

        y1, x2, y2, x1 = (
            y1 * 4,
            x2 * 4,
            y2 * 4,
            x1 * 4
        )

        cv2.rectangle(
            img,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        cv2.rectangle(
            img,
            (x1, y2 - 35),
            (x2, y2),
            (0, 255, 0),
            cv2.FILLED
        )

        cv2.putText(
            img,
            name,
            (x1 + 6, y2 - 6),
            cv2.FONT_HERSHEY_COMPLEX,
            1,
            (255, 255, 255),
            2
        )

        if name != "UNKNOWN":
            markAttendance(name)

    cv2.imshow('Webcam', img)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break


cap.release()
cv2.destroyAllWindows()

Loaded Names: ['amardeep', 'anil', 'istkar', 'Rajat', 'vipul', 'Yogesh']
Encoding Complete
Camera can't access
Failed to read from camera


: 